# ReazonSpeech — Japonca Altyazı (ONNX)

Japonca için en doğru açık kaynak ASR modeli.
NeMo framework gerektirmez — sherpa-onnx ile hafif inference.

**Pipeline:** `Video → ffmpeg (16kHz mono) → [Demucs vocal] → sherpa-onnx (ReazonSpeech) → SRT`

## Kullanım
A: Kurulum → B: Config + dosya → C: Transcript → D: SRT indir

---
# A) Kurulum

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}')

!pip install -q sherpa-onnx demucs soundfile
!apt-get -qq install ffmpeg > /dev/null 2>&1

# ReazonSpeech ONNX model indir
import os
MODEL_DIR = '/content/sherpa-onnx-nemo-parakeet-tdt_ctc-0.6b-ja-35000-int8'
if not os.path.exists(MODEL_DIR):
    print('\U0001f4e5 ReazonSpeech ONNX model indiriliyor...')
    !wget -q https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-nemo-parakeet-tdt_ctc-0.6b-ja-35000-int8.tar.bz2
    !tar xf sherpa-onnx-nemo-parakeet-tdt_ctc-0.6b-ja-35000-int8.tar.bz2
    !rm sherpa-onnx-nemo-parakeet-tdt_ctc-0.6b-ja-35000-int8.tar.bz2
    print('\u2705 Model indirildi')
else:
    print('\u2705 Model mevcut')

print('\u2705 Kurulum tamam')

---
# B) Config + Dosya

In [ ]:
USE_VOICE_ISOLATION = True

In [ ]:
import os
import subprocess
import time

from google.colab import files
from pathlib import Path

print('Dosya seç:')
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
STEM = Path(INPUT_FILE).stem
print(f'\u2705 {INPUT_FILE}')

# 16kHz mono WAV
AUDIO_FILE = 'audio_16k.wav'
subprocess.run(
    ['ffmpeg', '-y', '-i', INPUT_FILE, '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', AUDIO_FILE],
    capture_output=True, check=True,
)
print(f'\u2705 Ses: {AUDIO_FILE}')

In [ ]:
TRANSCRIPT_INPUT = AUDIO_FILE

if USE_VOICE_ISOLATION:
    print('\U0001f3b5 Vocal izolasyon (Demucs)...')
    DEMUCS_INPUT = 'audio_44k.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', INPUT_FILE, '-ar', '44100', '-ac', '2', DEMUCS_INPUT],
        capture_output=True, check=True,
    )
    t0 = time.time()
    result = subprocess.run(
        ['python', '-m', 'demucs', '--two-stems', 'vocals', '-n', 'htdemucs_ft', DEMUCS_INPUT],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        demucs_stem = os.path.splitext(os.path.basename(DEMUCS_INPUT))[0]
        vocal_path = f'separated/htdemucs_ft/{demucs_stem}/vocals.wav'
        if os.path.exists(vocal_path):
            subprocess.run(
                ['ffmpeg', '-y', '-i', vocal_path, '-ar', '16000', '-ac', '1', 'vocal_16k.wav'],
                capture_output=True, check=True,
            )
            TRANSCRIPT_INPUT = 'vocal_16k.wav'
            print(f'\u2705 Vocal izole ({time.time()-t0:.1f}s)')
        else:
            print('\u26a0\ufe0f Vocal bulunamadı, orijinal kullanılacak')
    else:
        print(f'\u26a0\ufe0f Demucs başarısız:\n{result.stderr[-300:]}')
else:
    print('Voice isolation kapalı')

---
# C) Transcript (sherpa-onnx)

In [ ]:
import time
import numpy as np

import sherpa_onnx
import soundfile as sf

# Model
recognizer = sherpa_onnx.OfflineRecognizer.from_nemo_ctc(
    model=f'{MODEL_DIR}/model.int8.onnx',
    tokens=f'{MODEL_DIR}/tokens.txt',
    num_threads=4,
    provider='cuda',
)

# Ses yükle
audio_data, sample_rate = sf.read(TRANSCRIPT_INPUT, dtype='float32')
assert sample_rate == 16000

# VAD ile segmentlere böl + her segment'i ayrı transcribe et
print(f'\U0001f3a4 Transcript: {TRANSCRIPT_INPUT}')
print('\U0001f50d VAD + segment transcribe...')

# VAD model indir
import urllib.request
VAD_MODEL = '/content/silero_vad.onnx'
if not os.path.exists(VAD_MODEL):
    urllib.request.urlretrieve('https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/silero_vad.onnx', VAD_MODEL)

vad_config = sherpa_onnx.VadModelConfig(
    silero_vad=sherpa_onnx.SileroVadModelConfig(
        model=VAD_MODEL,
        min_silence_duration=0.3,
        min_speech_duration=0.5,
    ),
    sample_rate=16000,
)

vad = sherpa_onnx.VoiceActivityDetector(vad_config, buffer_size_in_seconds=600)
vad.accept_waveform(audio_data)
vad.flush()

segments = []
t0 = time.time()

while not vad.empty():
    seg = vad.front
    start_sec = seg.start / 16000.0
    samples = np.array(seg.samples, dtype=np.float32)
    end_sec = start_sec + len(samples) / 16000.0

    s = recognizer.create_stream()
    s.accept_waveform(16000, samples)
    recognizer.decode_stream(s)
    text = s.result.text.strip()

    if text:
        segments.append({'start': start_sec, 'end': end_sec, 'text': text})

    vad.pop()

elapsed = time.time() - t0
print(f'\u2705 {len(segments)} segment, {elapsed:.1f}s')
for s in segments[:10]:
    print(f'  [{s["start"]:7.2f}-{s["end"]:7.2f}] {s["text"]}')

---
# D) SRT İndir

In [ ]:
from google.colab import files


def ts(sec):
    h, m, s, ms = int(sec//3600), int(sec%3600//60), int(sec%60), int(sec%1*1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'


tag = 'vocal' if USE_VOICE_ISOLATION else 'raw'
srt_path = f'{STEM}_reazonspeech_{tag}.srt'

with open(srt_path, 'w', encoding='utf-8') as f:
    for i, seg in enumerate(segments, 1):
        f.write(f'{i}\n{ts(seg["start"])} --> {ts(seg["end"])}\n{seg["text"]}\n\n')

print(f'\u2705 {srt_path} ({len(segments)} segment)')
files.download(srt_path)